# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Notebook authored and executed 3x on fresh CPU Colab runtimes; no single fresh run completed all 10 cells. Best attempt (logs/attempt1_output.ipynb) executed 9/10 cells including the complete Table S7 verification analysis (171 unique genes matched the paper's 171; 95 vs 93 CDS genes), the per-trait association graph, and the live DSpace re-check. It failed only on an R-install output-capture bug (fixed and verified in attempts 2-3, which installed all 6 R packages); attempts 2-3 then hit persistent transient bioRxiv 429 on the supplement download (bounded retry executed). Two evidence-led repairs spent and the 3-session budget exhausted, so execution stopped per the bounded-repair rule. Material core blocker, verified in every run and from DGX metadata: the ETH Research Collection dataset (handle 20.500.11850/809138) publicly enumerates only LICENSE and TEXT bundles, so the archived tables required by the authors' Pipeline B/C are not publicly retrievable; the guarded core raises Runt

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: From smartphone to gene: Automated image-based phenotyping captures flowering diversity of common buckwheat

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-920ac198d9ff0fb3b83c5af1baed4f1a)


# From smartphone to gene: flowering phenotyping in buckwheat — reproduction notebook

**Paper:** Carles Quesada-Traver, Fabian Hess, Flavian Tschurr, Lukas Kronenberg, Gabriele Rimat, Steven Yates, Bruno Studer (2026).
*From smartphone to gene: Automated image-based phenotyping captures flowering diversity of common buckwheat.*
bioRxiv 2026.11.03.763633 (v1, posted 2026-11-04), DOI [10.64898/2026.11.03.763633](https://doi.org/10.64898/2026.11.03.763633), CC BY 4.0.

**Author code:** [cquesadat/buckwheat-flowering](https://github.com/cquesadat/buckwheat-flowering), commit `0e47467eac97dcb447419994021e6d4c0ed44d88`
("Initial release"; the repository has **no LICENSE file**, so the code is used here with attribution and licensing remains unverified).

**Companion dataset:** ETH Zurich Research Collection, [handle 20.500.11850/809138](https://www.research-collection.ethz.ch/handle/20.500.11850/809138) (CC BY 4.0, ~285 GB).

## Scope of this notebook — PARTIAL, DATA-BLOCKED (read before running)

**Executed here (verified):**
1. Retrieval and SHA-256 verification of the pinned author analysis code, plus an R syntax-parse check.
2. Installation of the R packages used by the authors' flowering-dynamics pipeline (Pipeline B).
3. Download and parsing of the paper's official supplementary tables (`media-2.xlsx`, bioRxiv DC2).
4. A verification analysis that **re-derives the paper's reported candidate-gene counts from the authors' own
   published Table S7** (paper Section 3.6: 343 uniquely identified image-derived marker–trait associations,
   of which 265 co-localize with 171 annotated genes, 93 containing markers inside the putative CDS).
5. A per-trait association summary graph created from Table S7 (notebook-created visualization, not an author figure).
6. A live re-check of whether the companion dataset's archived analysis tables
   (`Results_Flowering_Perc_*.csv`, design/sowing/QC tables, fitted-parameter tables) are publicly downloadable.

**Blocked (guarded, not executed):** the core Pipeline B refit — re-running the authors' two-phase
flowering/abscission model fitting on the archived Flower % tables and the SpATS stage-1 genotype estimates.
The companion ETH dataset item exposes **only its LICENSE and README bundles** through its public API
(as of 2026-11-25), so the archived tables could not be retrieved. The final cell raises a `RuntimeError`
explaining this instead of silently producing a misleading result.

**Out of scope (documented):** Mask R-CNN segmentation (trained checkpoint `model-400000` is not confirmed
among the distributed assets), GAPIT GWAS reruns, and across-year REML genotype estimates (commercial ASReml-R).

*Notebook glue (downloads, staging, verification analysis) is AI-generated for this reproduction; the authors
did not provide this Colab implementation. Author R scripts are fetched and would be executed verbatim.*
*日本語: このノートブックは部分再現です。実行できるのは補足表の検証分析までで、本体データが公開されていないため
Pipeline B の再フィットは最後のセルで意図的にエラーになります。詳細は各説明を参照してください。*

## Runtime selection and how to run

**Select a CPU runtime** (Runtime → Change runtime type → CPU). No GPU is used anywhere in this
reproduction: the executed work is R statistical fitting code, table parsing and plotting. If the guarded
core becomes runnable (dataset published), it still needs only CPU. Expected duration on CPU: ~5–10 minutes
(R package installation dominates); downloads are a few megabytes. Run everything with Runtime → Run all.

*日本語: GPU は不要です。CPU ランタイムで「ランタイム → すべてのセルを実行」してください。所要時間は
パッケージインストールを含め約 5〜10 分、ダウンロードは数 MB 程度です。*

**Cell 1 — runtime inventory.** Reports the Python/R versions, CPU count and RAM of the allocated
Colab VM, and which of the pipeline's R packages are already installed. Expected: Python 3.12/3.13, R ≥ 4.4
preinstalled, most CRAN packages absent (installed in a later cell).
*日本語: Python/R のバージョンと CPU/メモリ、必要な R パッケージの有無を確認します。*

In [ ]:
import os, platform, subprocess, sys
print("Python:", sys.version.split()[0], "|", platform.platform())
print("CPUs:", os.cpu_count())
with open("/proc/meminfo") as f:
    for line in f:
        if line.startswith("MemTotal"):
            print("RAM:", line.strip()); break
r = subprocess.run(
    ["/usr/bin/Rscript", "-e",
     'cat(R.version.string, "\n"); ip <- rownames(installed.packages()); '
     'cat("preinstalled pipeline packages:", '
     'paste(intersect(c("dplyr","lubridate","ggplot2","cowplot","SpATS","scam"), ip), collapse=", "), "\n")'],
    capture_output=True, text=True, timeout=300)
print("Rscript exit:", r.returncode)
print(r.stdout.strip())
if r.returncode != 0:
    print("STDERR:", r.stderr[-1500:])

**Cell 2 — fetch the pinned author code.** Downloads the 9 files needed by the flowering-dynamics
pipeline (Pipeline B) plus both pipeline READMEs from the author repository at the pinned commit
`0e47467eac97`, directly from `raw.githubusercontent.com`, and verifies each file's byte size and SHA-256
against the manifest below (recorded from the same commit on 2026-11-25). Expected: 9 OK lines and no
verification error. The dataset itself is **not** fetched here.
*日本語: 論文の著者コード（コミット 0e47467eac97）を pinned の raw URL から取得し、サイズと SHA-256 を検証します。*

In [ ]:
import hashlib, pathlib, urllib.request

COMMIT = "0e47467eac97dcb447419994021e6d4c0ed44d88"
MANIFEST = {'README.md': (7750, '4469b81159876b7a3b93084e8fbb0bb463d3653cd25bab3c5caea9a2c81bace6'), 'B_Flowering_dynamics_Modelling/README.md': (2374, 'e2a944c6ac09eb92ab7957555a74cca0631922216bb3a90e9c0e84b925cee7a8'), 'C_Genome_Wide_Analysis_Studies/README.md': (1891, '301c89a590854d34ba247816dbf196aee5341855e8df13532a20b69c2d445ded'), 'B_Flowering_dynamics_Modelling/0.1_Add_DAS.R': (2307, 'f3560c0c13d984d2e7dc982ffec7633e885814b7eb3366cec355b6233dfa1ee3'), 'B_Flowering_dynamics_Modelling/1_Flowering_Model_Fitting.R': (2795, '3e970d943ed1485880f4e04323c1a63c7557503b9a57f74a48c2d6ee96c8c804'), 'B_Flowering_dynamics_Modelling/1.5_Filter_and_Convert_to_DAS.R': (6098, '75a66e1894d788cb54a1c0286917706226a0cf0c8206d8a994901bd5ba7bc0b0'), 'B_Flowering_dynamics_Modelling/2_BLUEs_BLUPs_all_experiments_and_BBCH.R': (17308, '282a9846906f3d50e9596d3a110dacdc03b792028e93f7f6a5d46d7a84ad362f'), 'B_Flowering_dynamics_Modelling/utils/FUN_model_fitting.R': (15344, 'de26b65b39cde0247622d1486520bc0097d9b72e210783662f35c7d5f977b5a2'), 'B_Flowering_dynamics_Modelling/utils/FUN_utils.R': (989, '6d7eaf68f3fec2f16c7969c27ec03b5a272ea21cc654e03b07eb51974c91ffdf')}
BASE = f"https://raw.githubusercontent.com/cquesadat/buckwheat-flowering/{COMMIT}"
ROOT = pathlib.Path("/content/buckwheat-flowering")

for rel, (size, sha) in MANIFEST.items():
    with urllib.request.urlopen(f"{BASE}/{rel}", timeout=120) as r:
        data = r.read()
    assert len(data) == size, f"size mismatch for {rel}: {len(data)} vs {size}"
    digest = hashlib.sha256(data).hexdigest()
    assert digest == sha, f"sha256 mismatch for {rel}: {digest}"
    dest = ROOT / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(data)
    print(f"OK {rel} ({size} bytes)")

**Cell 3 — R syntax check of the fetched scripts.** Parses all fetched `.R` files with R's own
parser (`parse()`), confirming the author code is syntactically intact as retrieved. Expected: one
"parsed OK" line per file, exit status 0.
*日本語: 取得した著者の R スクリプトを R のパーサで構文確認します（実行はまだしません）。*

In [ ]:
import subprocess
R_FILES = [
    "B_Flowering_dynamics_Modelling/0.1_Add_DAS.R",
    "B_Flowering_dynamics_Modelling/1_Flowering_Model_Fitting.R",
    "B_Flowering_dynamics_Modelling/1.5_Filter_and_Convert_to_DAS.R",
    "B_Flowering_dynamics_Modelling/2_BLUEs_BLUPs_all_experiments_and_BBCH.R",
    "B_Flowering_dynamics_Modelling/utils/FUN_model_fitting.R",
    "B_Flowering_dynamics_Modelling/utils/FUN_utils.R",
]
paths = [f"/content/buckwheat-flowering/{f}" for f in R_FILES]
r = subprocess.run(
    ["/usr/bin/Rscript", "-e",
     'for (f in commandArgs(TRUE)) { parse(f); cat("parsed OK:", f, "\n") }'] + paths,
    capture_output=True, text=True, timeout=300)
print(r.stdout)
print("exit:", r.returncode)
if r.returncode != 0:
    print("STDERR:", r.stderr[-2000:])
    raise RuntimeError("Author R scripts failed the syntax check")

**Cell 4 — install the Pipeline B R dependencies.** Installs the CRAN packages the authors'
flowering-dynamics pipeline needs: `dplyr` and `lubridate` (data handling and dates), `ggplot2`/`cowplot`
(used inside the fitting helper), `SpATS` (stage-1 spatial genotype estimates), and `scam` (monotonic
p-spline BBCH interpolation of paper Section 2.2). Expected: streamed "installing … / done" lines and a
final availability list. Package installation is prepared here so the guarded core (last cell) is ready to
run as soon as the companion dataset becomes publicly available.
*日本語: Pipeline B が使う R パッケージ（dplyr, lubridate, ggplot2, cowplot, SpATS, scam）を CRAN から
インストールします。数分かかります。*

In [ ]:
r = subprocess.run(
    ["/usr/bin/Rscript", "-e", (
        'options(repos = c(CRAN = "https://cloud.r-project.org"))\n'
        'need <- c("dplyr","lubridate","ggplot2","cowplot","SpATS","scam")\n'
        'ip <- rownames(installed.packages())\n'
        'miss <- setdiff(need, ip)\n'
        'cat("to install:", paste(miss, collapse = ", "), "\n")\n'
        'for (p in miss) {\n'
        '  cat("installing", p, "...\n"); flush.console()\n'
        '  install.packages(p)\n'
        '  cat("done", p, "\n"); flush.console()\n'
        '}\n'
        'ip2 <- rownames(installed.packages())\n'
        'cat("available after install:", paste(intersect(need, ip2), collapse = ", "), "\n")\n'
        'cat("still missing:", paste(setdiff(need, ip2), collapse = ", "), "\n")')],
    timeout=2700, capture_output=True, text=True)
print(r.stdout[-5000:] if r.stdout else "(no stdout)")
print("exit:", r.returncode)
if r.returncode != 0:
    print("STDERR tail:", r.stderr[-2000:])
    print("NOTE: a failed install does not abort the notebook here; the guarded core re-checks what it needs.")

**Cell 5 — download the official supplementary tables.** Fetches `media-2.xlsx` (Supplementary
Tables S1–S7) from the paper's own supplementary-material tab on bioRxiv and saves it under `/content`.
This is a data download and is performed here in Colab, never on the authoring host. Expected: HTTP 200,
a `PK` (zip) signature, ~105 KB (105,525 bytes observed on 2026-11-25; a small difference only triggers a
warning), and a SHA-256 fingerprint printed for provenance. If bioRxiv rate-limits the request
(HTTP 429, observed transiently on 2026-11-25), the cell waits once for 60 s and retries.
*日本語: 論文の補足表（media-2.xlsx）を Colab 内にダウンロードし、署名・サイズ・SHA-256 を確認します。
429 の場合は 60 秒待って 1 回だけ再試行します。*

In [ ]:
import hashlib, pathlib, time, urllib.error, urllib.request

SUPP = ("https://www.biorxiv.org/content/biorxiv/early/2026/11/04/"
        "2026.11.03.763633/DC2/embed/media-2.xlsx?download=true")
OUT = pathlib.Path("/content/supplements/media-2.xlsx")
OUT.parent.mkdir(parents=True, exist_ok=True)
data = None
for attempt in (1, 2):
    req = urllib.request.Request(SUPP, headers={"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) Chrome/126.0"})
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            data = r.read()
            print("HTTP", r.status, "| bytes:", len(data), "| final URL:", r.geturl(), f"(attempt {attempt})")
        break
    except urllib.error.HTTPError as e:
        print(f"HTTP {e.code} (attempt {attempt})")
        if e.code == 429 and attempt == 1:
            time.sleep(60)   # bounded wait: bioRxiv rate limiting is transient
        else:
            raise
assert data is not None
assert data[:2] == b"PK", "not an xlsx (zip) payload"
if len(data) != 105525:
    print(f"WARNING: size {len(data)} differs from 105,525 observed on 2026-11-25")
digest = hashlib.sha256(data).hexdigest()
print("sha256:", digest)
REFERENCE_SHA256 = "4fad153dee3f57281886d3e033028e9264e95297c651922111f92b51387e1884"
if digest != REFERENCE_SHA256:
    print("WARNING: sha256 differs from the copy downloaded during the 2026-11-25 validation runs "
          "(bioRxiv may have replaced the file; proceeding, but re-check the tables)")
OUT.write_bytes(data)

**Cell 6 — parse and inspect the supplement tables.** Opens the workbook and shows the Index sheet,
Table S1 (definitions of the 22 image-derived + manual traits) and Table S2 (chromosome-specific LD
windows used by the paper's LD-block analysis). Data tables are read below their header row with blank
rows dropped. Expected: the Index listing S1–S7, ~25 trait rows in S1 and 8–9 chromosome rows in S2.
*日本語: 補足表を読み込み、Index・Table S1（形質定義）・Table S2（染色体別 LD ウィンドウ）を表示します。*

In [ ]:
try:
    import openpyxl, pandas as pd
except ImportError:
    import subprocess as sp, sys as s
    sp.run([s.executable, "-m", "pip", "install", "-q", "openpyxl"], check=True)
    import openpyxl, pandas as pd

SHEETS = pd.read_excel("/content/supplements/media-2.xlsx", sheet_name=None, header=None)
print("sheets:", list(SHEETS))

def load_table(name, header_row=1):
    raw = SHEETS[name]
    cols = [str(h).strip() for h in raw.iloc[header_row].tolist()]
    data = raw.iloc[header_row + 1:].dropna(how="all")
    data.columns = cols
    return data.reset_index(drop=True)

print("=== Index sheet ===")
print(SHEETS["Index"].dropna(how="all").to_string(index=False))

s1 = load_table("TableS1")
s2 = load_table("TableS2 ")
s7 = load_table("TableS7")
print(f"\n=== Table S1 — trait definitions ({len(s1)} rows) ===")
print(s1.to_string(index=False, max_colwidth=60))
print(f"\n=== Table S2 — chromosome-specific LD windows ({len(s2)} rows) ===")
print(s2.to_string(index=False))

**Cell 7 — verification analysis: re-derive the paper's candidate-gene counts from Table S7.**
Paper Section 3.6 states: *“The genetic context of the Image-derived phenotype associations was examined
in detail using the uniquely identified 343 marker-trait associations. Of these, 265 were found to
co-localize with 171 annotated genes, 93 of which contained markers located within the putative coding
sequence (CDS; Table 2).”* This cell computes the corresponding statistics directly from the authors'
published Table S7 (SNP, Trait, Group, Gene, SNP-in-CDS) and compares them with the paper-reported
values. Because a table row can differ from one association (e.g. a SNP overlapping several genes), both
raw row counts and uniqueness-collapsed counts are reported; the comparison table states explicitly which
quantity each row measures and whether it matches. Expected: values close to 343/265/171/93, with any
deviation reported honestly rather than forced.
*日本語: 著者公開の Table S7 から、論文 §3.6 が報告する候補遺伝子関連の集計（343/265/171/93）を再計算して
照合します。行数と一意カウントの両方を示し、一致・不一致をそのまま報告します。*

In [ ]:
import pandas as pd
s7 = s7.rename(columns={"SNP in CDS": "SNP_in_CDS"})
assoc = s7[["SNP", "Trait", "Group", "Gene", "SNP_in_CDS"]].copy()
print("Group values:", sorted(assoc["Group"].dropna().astype(str).str.strip().unique()))
print("SNP_in_CDS values:", sorted(assoc["SNP_in_CDS"].dropna().astype(str).str.strip().unique()))

gene_split = (assoc["Gene"].dropna().astype(str)
              .str.split(r"[;,]", regex=True)
              .explode().str.strip())
gene_split = gene_split[~gene_split.str.lower().isin(["", "na", "nan"])]

is_cds = assoc["SNP_in_CDS"].astype(str).str.strip().str.lower().isin(["yes", "true", "1", "y"])
cds_gene_split = (assoc.loc[is_cds, "Gene"].dropna().astype(str)
                  .str.split(r"[;,]", regex=True).explode().str.strip())
cds_gene_split = cds_gene_split[~cds_gene_split.str.lower().isin(["", "na", "nan"])]

computed = {
    "rows listed in Table S7": len(assoc),
    "unique (SNP, Trait) associations": assoc[["SNP", "Trait"]].drop_duplicates().shape[0],
    "unique SNPs": assoc["SNP"].nunique(),
    "rows co-localizing with annotated genes": int(assoc["Gene"].notna().sum()),
    "unique annotated genes (split multi-gene cells)": gene_split.nunique(),
    "unique genes containing markers in CDS": cds_gene_split.nunique(),
}
paper = {
    "rows listed in Table S7": ("table listing (no exact paper claim)", None),
    "unique (SNP, Trait) associations": ("unique image-derived MTAs examined (§3.6)", 343),
    "unique SNPs": ("not directly reported", None),
    "rows co-localizing with annotated genes": ("associations co-localizing with genes (§3.6)", 265),
    "unique annotated genes (split multi-gene cells)": ("annotated genes (§3.6)", 171),
    "unique genes containing markers in CDS": ("genes with markers in CDS (§3.6)", 93),
}
cmp_rows = []
for k, v in computed.items():
    label, expected = paper[k]
    cmp_rows.append({"quantity": k, "re-derived from Table S7": v,
                     "paper-reported quantity": label, "paper value": expected,
                     "match": ("—" if expected is None else "YES" if v == expected else "NO")})
comparison = pd.DataFrame(cmp_rows)
print(comparison.to_string(index=False))

for gene in ["ARF2", "EIN2", "MDN1", "DPH1"]:
    hits = sorted(set(g for g in gene_split.unique() if gene.upper() in g.upper()))
    print(f"highlighted gene family {gene}: {hits if hits else 'not present in Table S7 Gene column'}")

**Cell 8 — graph of gene-overlapping associations per trait.** A notebook-created visualization
(not an author figure): number of Table S7 associations per trait, stacked by trait group, from the
actual downloaded supplement data. It shows which flowering traits carry the most genetic signal in the
paper's candidate-gene analysis. The figure is saved as PNG under `/content/out/` and displayed.
*日本語: Table S7 の関連数を形質ごと（グループ別に積み上げ）で可視化します。これは本ノートブック独自の
補足図であり、論文の図ではありません。*

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pathlib

ct = pd.crosstab(assoc["Trait"], assoc["Group"].fillna("(unlabelled)"))
ax = ct.plot.barh(stacked=True, figsize=(12, 6.75), color=plt.cm.tab10.colors)
ax.set_xlabel("gene-overlapping marker-trait associations listed in Table S7")
ax.set_ylabel("trait")
ax.set_title("Marker-trait associations per trait, by trait group\n"
             "(re-derived from the paper's Table S7; notebook-created visualization)")
ax.legend(title="trait group", fontsize=8)
plt.tight_layout()
pathlib.Path("/content/out").mkdir(exist_ok=True)
plt.savefig("/content/out/table_s7_associations_per_trait.png", dpi=100)
plt.show()

**Cell 9 — re-check the companion dataset's public availability.** Queries the ETH Research
Collection's documented public REST API for the dataset item (handle 20.500.11850/809138) and lists the
bundles visible to an anonymous client, with one bounded retry if the service rate-limits us. On DGX-side
metadata inspection (2026-11-25) only `LICENSE` and `TEXT` (README) bundles were enumerable, so the
archived analysis tables are expected to remain unavailable; if additional payload bundles appear, the
required small CSVs (`Results_Flowering_Perc_ETH23.csv`, `Design_ETH23.csv`, `sowing_dates.csv`,
`manual_exclusions.csv`, `parameters_clean_das_2023_ETH.csv`) are downloaded into `/content` and
`DATA_AVAILABLE` is set to True.
*日本語: ETH Research Collection の公開 API で、データセット本体（アーカイブ済み解析テーブル）が匿名で
取得可能かを再確認します。取得できない場合は DATA_AVAILABLE=False となり、次のセルが理由を明示します。*

In [ ]:
import json, time, urllib.error, urllib.request

ITEM = "854f9f6f-859d-4bc4-b630-f200ff6edd81"
BUNDLES_URL = f"https://www.research-collection.ethz.ch/server/api/core/items/{ITEM}/bundles"
NEEDED = {
    "Results_Flowering_Perc_ETH23.csv": r"Results_Flowering_Perc_ETH23\.csv$",
    "Design_ETH23.csv": r"Design_ETH23\.csv$",
    "sowing_dates.csv": r"sowing_dates\.csv$",
    "manual_exclusions.csv": r"manual_exclusions\.csv$",
    "parameters_clean_das_2023_ETH.csv": r"parameters_clean_das_2023_ETH\.csv$",
}

def get_json(url):
    req = urllib.request.Request(url, headers={"Accept": "application/json",
                                               "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) Chrome/126.0"})
    with urllib.request.urlopen(req, timeout=90) as r:
        return json.load(r)

bundles = None
for attempt in (1, 2):
    try:
        bundles = get_json(BUNDLES_URL)
        break
    except urllib.error.HTTPError as e:
        print(f"HTTP {e.code} on bundles listing (attempt {attempt})")
        if e.code == 429 and attempt == 1:
            time.sleep(60)
        else:
            break

if bundles is None:
    DATA_AVAILABLE = False
    AVAILABILITY_NOTE = "inconclusive: the public API rate-limited this re-check (HTTP 429)"
else:
    names = [b["name"] for b in bundles["_embedded"]["bundles"]]
    print("bundles visible to anonymous clients:", names)
    payload = [b for b in bundles["_embedded"]["bundles"] if b["name"] not in ("LICENSE", "TEXT")]
    if payload:
        import pathlib, re
        TARGETS = {p: pathlib.Path("/content/payload") for p in NEEDED}
        for b in payload:
            listed = get_json(b["_links"]["bitstreams"]["href"])
            for bs in listed["_embedded"]["bitstreams"]:
                name = bs["name"]
                hit = next((k for k, pat in NEEDED.items() if re.search(pat, name)), None)
                if hit:
                    dest = pathlib.Path("/content/payload") / hit
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    with urllib.request.urlopen(bs["_links"]["content"]["href"], timeout=180) as r:
                        dest.write_bytes(r.read())
                    print("downloaded:", name, "->", dest)
        have = [k for k in NEEDED if (pathlib.Path("/content/payload") / k).exists()]
        print("required tables retrieved:", have)
        DATA_AVAILABLE = len(have) == len(NEEDED)
        AVAILABILITY_NOTE = f"payload bundles visible: {[b['name'] for b in payload]}"
    else:
        DATA_AVAILABLE = False
        AVAILABILITY_NOTE = ("anonymous API enumerates only LICENSE and TEXT bundles, matching the "
                             "DGX metadata inspection of 2026-11-25; the ~285 GB payload is not publicly listed")
print("DATA_AVAILABLE:", DATA_AVAILABLE)
print("availability note:", AVAILABILITY_NOTE)

**Cell 10 — guarded core: Pipeline B refit on the archived Flower % table (authors' scripts,
executed verbatim when data is available).** When `DATA_AVAILABLE` is True, this cell stages the ETH23
files into the authors' expected directory layout, runs their scripts `0.1_Add_DAS.R` (days after
sowing), `1_Flowering_Model_Fitting.R` (three-parameter logistic fit to the pre-peak flowering phase with
a 2 % coverage onset threshold, plus a linear fit to the abscission phase — 22 traits per plot), and
`1.5_Filter_and_Convert_to_DAS.R` (quality control: R² < 0.5 exclusion, manual exclusions, event dates →
DAS), then runs the authors' SpATS stage-1 BLUEs/BLUPs functions for one trait. While the companion
dataset payload is not publicly retrievable, the cell raises a `RuntimeError` with the precise blocker so
that Run all cannot silently produce a misleading result.
*日本語: コア解析。データが入手可能な場合は著者の Pipeline B スクリプトをそのまま実行します。現時点では
データセット本体が公開されていないため、誤った結果を避けるために RuntimeError を発生させて停止します。*

In [ ]:
if not DATA_AVAILABLE:
    raise RuntimeError(
        "Core Pipeline B demonstration blocked: the companion ETH Research Collection dataset "
        "(item 854f9f6f-859d-4bc4-b630-f200ff6edd81, handle 20.500.11850/809138, CC BY 4.0) does not "
        "publicly expose its ~285 GB payload through its documented REST API — as of 2026-11-25 only its "
        "LICENSE and TEXT (README) bundles are enumerable for anonymous clients, so the archived tables "
        "required by the authors' scripts (Results_Flowering_Perc_ETH23.csv, Design_ETH23.csv, "
        "sowing_dates.csv, manual_exclusions.csv, parameters_clean_das_2023_ETH.csv) could not be "
        "downloaded. Re-run this notebook when the dataset becomes publicly available; this cell will then "
        "execute the authors' Pipeline B scripts (commit 0e47467e) verbatim on ETH23 and the SpATS "
        "stage-1 genotype estimates."
    )

# --- Data available: stage the authors' directory layout and run their scripts verbatim ---
import pathlib, shutil, subprocess

ROOT = pathlib.Path("/content/buckwheat-flowering")
PAYLOAD = pathlib.Path("/content/payload")
B = ROOT / "B_Flowering_dynamics_Modelling"
(B / "data").mkdir(parents=True, exist_ok=True)
(B / "utils").mkdir(parents=True, exist_ok=True)          # author utils already fetched (Cell 2)
(B / "flowering_parameters").mkdir(parents=True, exist_ok=True)
A_ETH23 = ROOT / "A_Image_Segmentation_Modelling" / "Experiments" / "ETH23"
A_ETH23.mkdir(parents=True, exist_ok=True)

shutil.copy(PAYLOAD / "Results_Flowering_Perc_ETH23.csv", A_ETH23 / "Results_Flowering_Perc_ETH23.csv")
shutil.copy(PAYLOAD / "Results_Flowering_Perc_ETH23.csv", B / "data" / "Results_Flowering_Perc_ETH23.csv")
for f in ("sowing_dates.csv", "manual_exclusions.csv"):
    shutil.copy(PAYLOAD / f, B / "data" / f)
archived = PAYLOAD / "parameters_clean_das_2023_ETH.csv"
if archived.exists():
    shutil.copy(archived, B / "flowering_parameters" / "parameters_clean_das_2023_ETH.csv")

def run_r(script, timeout=1800):
    print(f"=== Rscript {script} ===")
    r = subprocess.run(["/usr/bin/Rscript", script], cwd=B, timeout=timeout,
                       capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print("STDERR tail:", r.stderr[-2000:])
        raise RuntimeError(f"author script failed: {script} (exit {r.returncode})")

run_r("0.1_Add_DAS.R data/Results_Flowering_Perc_ETH23.csv")   # adds days-after-sowing
run_r("1_Flowering_Model_Fitting.R")                            # two-phase fits per plot
run_r("1.5_Filter_and_Convert_to_DAS.R")                        # QC + event dates -> DAS

clean = B / "flowering_parameters" / "parameters_ETH23_clean_das.csv"
params = pd.read_csv(clean)
print("refitted plots:", len(params), "| traits per plot:", params.shape[1] - 2)
print(params.describe().T[["mean", "min", "max"]].head(8))

# SpATS stage-1 for the first cleaned trait, using the authors' own function definitions
first_trait = next(c for c in params.columns if c not in ("Plot", "ID"))
r_spats = subprocess.run(
    ["/usr/bin/Rscript", "-e", (
        'src <- readLines("2_BLUEs_BLUPs_all_experiments_and_BBCH.R")\n'
        'stop_at <- grep("^## Processing blocks", src)[1]\n'
        'eval(parse(text = paste(src[seq_len(stop_at - 1)], collapse = "\n")))\n'
        'design <- read.csv("data/Design_ETH23.csv")\n'
        'design$Plot <- paste0("Plot", design$ROW, "_", design$RANGE)\n'
        'param_data <- read.csv("flowering_parameters/parameters_ETH23_clean_das.csv")\n'
        f'one_param <- merge(design, param_data[, c("Plot", "ID", "{first_trait}")], by = "Plot")\n'
        f'BLUEs <- spats_BLUEs(one_param, "{first_trait}")\n'
        f'BLUPs <- spats_BLUPs(one_param, "{first_trait}")\n'
        'cat("SpATS stage-1 done for", "'+first_trait+'", "\n")\n'
        'cat("genotypes:", nrow(BLUEs), "| heritability:", round(BLUPs$h2[1], 3), "\n")')],
    cwd=B, timeout=1800, capture_output=True, text=True)
print(r_spats.stdout[-2000:])
if r_spats.returncode != 0:
    print("STDERR tail:", r_spats.stderr[-2000:])
    raise RuntimeError("SpATS stage-1 failed")


## Interpretation, differences from the paper, and validation record

- **Executed and verified in this notebook:** retrieval of the pinned author code (SHA-256 checked),
  R syntax parsing, Pipeline B dependency installation, download of the official supplement tables, and
  the Table S7 verification analysis comparing re-derived counts against the paper-reported values
  (343 unique image-derived associations; 265 co-localizing with 171 genes; 93 with CDS markers).
  Any mismatch between the re-derived and reported numbers is shown as `NO` in the comparison table above.
- **Blocked by data availability, not by code:** the core two-phase flowering/abscission refit and the
  SpATS stage-1 genotype estimates faithfully re-use the authors' own scripts, but the companion ETH
  dataset item does not publicly expose its payload (only LICENSE + README bundles were enumerable via
  its documented public API on 2026-11-25, re-checked live in Cell 9). The paper's data statement says
  code and data are available "upon reasonable request". If the dataset is published, re-running this
  notebook executes the full guarded path on CPU.
- **Out of scope:** Mask R-CNN segmentation (checkpoint `model-400000` not confirmed among distributed
  assets), GAPIT GWAS reruns, across-year REML (commercial ASReml-R), and the manually scored BBCH
  p-spline interpolation (raw scores are only in the dataset payload; supplement Table S6 holds summary
  statistics only).
- **Provenance:** paper CC BY 4.0 (bioRxiv 2026.11.03.763633); companion dataset CC BY 4.0 (ETH Zurich
  Research Collection); author code commit `0e47467e…` (no LICENSE file — attribution given, redistribution
  of the fetched scripts is not asserted); supplementary tables as published on the paper's bioRxiv tab.
  The Table S7 figure above is a notebook-created visualization, not an author figure.
- *日本語: 本ノートブックは、著者コードの取得・検証、補足表の検証分析（Table S7 から論文記載の集計値の
  再導出）までを実行します。コアの開花動態モデル再フィットは、伴随データセットが公開されていないため
  意図的にブロックされています。データが公開されれば、同じノートブックを CPU で再実行すれば全文脈が動きます。*

**References:** bioRxiv preprint DOI 10.64898/2026.11.03.763633 (v1, 2026-11-04); author repository
`cquesadat/buckwheat-flowering` commit `0e47467eac97dcb447419994021e6d4c0ed44d88`; ETH Research Collection
handle 20.500.11850/809138; PhenoPaper investigation `p-920ac198d9ff0fb3b83c5af1baed4f1a-20261125T183632Z-2181608`
(used as prior research guidance, not primary proof).